# Lecture 30.02: Gradient Boosting Classifier (Intuition) — Log-Odds & Sigmoid Residuals

This notebook contains runnable code aligned with the video lecture demonstration.


In [ ]:
import numpy as np

def sigmoid(z):
    """Numerically stable logistic sigmoid function."""
    return np.where(z >= 0, 1.0 / (1.0 + np.exp(-z)), np.exp(z) / (1.0 + np.exp(z)))

def gradient_boosting_binary_classifier_scratch(X, y, n_estimators=5, learning_rate=0.2):
    """
    First-principles demonstration of Gradient Tree Boosting for Binary Classification.
    Operates in Log-Odds space using the logistic sigmoid loss.
    """
    X_arr = np.asarray(X, dtype=float)
    y_arr = np.asarray(y, dtype=float)
    
    # 1. Baseline prior probability and log-odds F0
    p_prior = np.mean(y_arr)
    # Clip to avoid division by zero
    p_prior = np.clip(p_prior, 1e-6, 1.0 - 1e-6)
    f0 = float(np.log(p_prior / (1.0 - p_prior)))
    
    current_log_odds = np.full_like(y_arr, fill_value=f0)
    
    # 2. Sequential Boosting Stages
    from sklearn.tree import DecisionTreeRegressor
    trees = []
    
    for stage in range(n_estimators):
        # Convert current log-odds to probability
        current_probs = sigmoid(current_log_odds)
        
        # Residuals = y - p
        residuals = y_arr - current_probs
        
        # Fit regression tree on pseudo-residuals
        tree = DecisionTreeRegressor(max_depth=1, random_state=42 + stage)
        tree.fit(X_arr, residuals)
        trees.append(tree)
        
        # Update log-odds via shrinkage
        current_log_odds += learning_rate * tree.predict(X_arr)
        
    final_probs = sigmoid(current_log_odds)
    predicted_classes = (final_probs >= 0.5).astype(int)
    
    return f0, trees, final_probs, predicted_classes

# ----------------- Driver Execution & Testing -----------------
if __name__ == "__main__":
    X_sample = np.array([
        [22.0, 25.0], [45.0, 55.0], [52.0, 70.0],
        [20.0, 22.0], [38.0, 48.0], [60.0, 85.0]
    ])
    y_sample = np.array([0, 1, 1, 0, 0, 1])  # 0 = Low Risk, 1 = High Risk
    
    f0_init, trained_trees, probas, preds = gradient_boosting_binary_classifier_scratch(
        X_sample, y_sample, n_estimators=6, learning_rate=0.3
    )
    
    print("=" * 60)
    print("--- GRADIENT BOOSTING CLASSIFIER (LOG-ODDS SCRATCH) ---")
    print(f"Initial Baseline Log-Odds (F0) : {f0_init:.4f}")
    print(f"Actual Target Classes          : {y_sample.tolist()}")
    print(f"Final Model Probabilities      : {[round(p, 3) for p in probas]}")
    print(f"Predicted Discrete Classes     : {preds.tolist()}")
    print(f"Classification Accuracy        : {np.mean(preds == y_sample) * 100:.1f}%")
    print("=" * 60)
